# VC Dimension, NIP, and the Learnability of Deep Computations
## From Shattering to the Fundamental Theorem of PAC Learning

**Companion notebook** — *Complexity of Deep Computations via Topology* [Dueñez, Iovino, Matos-Wiederhold, Salvetti, Tall]

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/public-site/blob/main/2026-FES_Acatlan-ultracomputos/notebooks/vc_nip.ipynb)

---

The **NIP** (No Independence Property) is the condition that distinguishes
*tame* from *wild* deep computations in the research papers.
In the finitary setting of machine learning, NIP is equivalent to a classical concept:

> **A function class is NIP ↔ it has finite VC dimension ↔ it is PAC-learnable.**

This notebook builds the VC dimension concept from scratch through interactive
visualizations, then makes the NIP connection explicit via the **independence
matrix** — the combinatorial object at the heart of both the BFT theorem and
modern ML theory.

The final section connects this to **in-context learning** in large language models —
an active research frontier where the same VC-theoretic ideas are now playing a central role.


## VC dimension: can a function class label any pattern?

Let $\mathcal{F}$ be a class of functions $f : X \to \{-1, +1\}$ (binary classifiers).

**Shattering.** A finite set $S = \{x_1, \ldots, x_n\} \subset X$ is *shattered* by $\mathcal{F}$ if
for **every** labeling $\sigma : S \to \{-1,+1\}$, there exists $f \in \mathcal{F}$ with
$f(x_i) = \sigma(x_i)$ for all $i$.
Concretely: $\mathcal{F}$ can realize all $2^n$ possible labelings of $S$.

**VC dimension.** $\operatorname{VC-dim}(\mathcal{F}) = \sup\{|S| : \mathcal{F} \text{ shatters } S\}$.

| Function class | VC dimension |
|---------------|:---:|
| Threshold functions $\mathbf{1}_{x \geq t}$ on $\mathbb{R}$ | 1 |
| Intervals $\mathbf{1}_{a \leq x \leq b}$ on $\mathbb{R}$ | 2 |
| Half-planes in $\mathbb{R}^2$ | 3 |
| Neural net with $p$ parameters (ReLU) | $O(p \log p)$ |
| All binary functions on $X$ | $\infty$ |

**Finite VC dim = NIP.** The key theorem (Shelah / Blumer et al.):
a class $\mathcal{F}$ has finite VC dimension if and only if it has the NIP —
no infinite independent pattern exists (no infinite set is shattered).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import ipywidgets as widgets
from scipy.optimize import linprog
%matplotlib inline


In [ ]:
# ── Linear separability (LP test) ────────────────────────────────────────────

def is_separable(points, labels):
    # labels in {+1, -1}
    # Returns (separable, w, b) where the separating hyperplane is w @ x + b = 0
    pts  = np.asarray(points, dtype=float)
    labs = np.asarray(labels, dtype=float)
    n    = len(labs)
    A_ub = np.column_stack([-labs * pts[:, 0], -labs * pts[:, 1], -labs])
    b_ub = -np.ones(n)
    res  = linprog(np.zeros(3), A_ub=A_ub, b_ub=b_ub,
                   bounds=[(-1e4, 1e4)] * 3, method='highs')
    if res.status == 0:
        return True, res.x[:2], res.x[2]
    return False, None, None


def separating_line_pts(w, b, xlim, n=200):
    # Return (xs, ys) for the separating line w[0]*x + w[1]*y + b = 0
    if abs(w[1]) > abs(w[0]):
        xs = np.linspace(xlim[0], xlim[1], n)
        ys = -(w[0] * xs + b) / w[1]
    else:
        ys = np.linspace(xlim[0], xlim[1], n)
        xs = -(w[1] * ys + b) / w[0]
    return xs, ys


# ── Threshold function shattering on R ───────────────────────────────────────

def threshold_achievable(n_pts):
    # Points at x = 0, 1, ..., n_pts-1
    # Threshold function: +1 if x >= t, -1 otherwise
    # Returns list of (labeling, achievable) tuples
    n = n_pts
    results = []
    for mask in range(1 << n):
        labels = [+1 if (mask >> i) & 1 else -1 for i in range(n)]
        # A threshold function achieves labels iff there exists t such that:
        # label[i] = +1 <=> i >= t (as an integer threshold between grid points)
        # Equivalent: labels form a suffix of +1s (all -1s before some index, all +1s after)
        achievable = False
        for t in range(n + 1):   # t = 0 means all +1, t = n means all -1
            pred = [+1 if i >= t else -1 for i in range(n)]
            if pred == labels:
                achievable = True
                break
        results.append((labels, achievable))
    return results


# ── Independence matrix ───────────────────────────────────────────────────────

def interval_matrix(test_points, a_vals, b_vals):
    # Entry (i, j) = 1 if test_points[i] in [a_vals[j], b_vals[j]]
    pts = np.array(test_points)[:, np.newaxis]
    a   = np.array(a_vals)[np.newaxis, :]
    b   = np.array(b_vals)[np.newaxis, :]
    return ((pts >= a) & (pts <= b)).astype(float)


def all_binary_matrix(n):
    # All 2^n binary patterns: entry (i, j) = j-th bit of i
    n_cols = 1 << n
    rows   = np.arange(n)[:, np.newaxis]
    cols   = np.arange(n_cols)[np.newaxis, :]
    return ((cols >> rows) & 1).astype(float)


## Warmup: threshold functions on $\mathbb{R}$

The simplest possible binary classifiers: $f_t(x) = \operatorname{sign}(x - t)$.
Every function in this class is a "step" — positive on one side of a threshold, negative on the other.

**Can $\{f_t\}$ shatter a 1-point set $\{x_1\}$?**
Yes: choose $t < x_1$ for label $+1$, or $t > x_1$ for label $-1$.

**Can $\{f_t\}$ shatter a 2-point set $\{x_1, x_2\}$ with $x_1 < x_2$?**
No: the labeling $(+1, -1)$ would require $t \leq x_1$ (for $x_1 = +1$)
AND $t > x_2$ (for $x_2 = -1$), which is impossible since $x_1 < x_2$.

**Therefore**: $\operatorname{VC-dim}(\{f_t\}) = 1$.

The interactive plot below visualizes the achievable labelings for $n = 1, 2, 3$ points.


In [ ]:
@widgets.interact(
    n_pts=widgets.IntSlider(
        value=2, min=1, max=4, step=1,
        description='Number of points n',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px'),
    ),
    threshold=widgets.FloatSlider(
        value=1.5, min=-0.5, max=4.5, step=0.1,
        description='Threshold t',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px'),
    ),
)
def threshold_plot(n_pts, threshold):
    results = threshold_achievable(n_pts)
    n_total = 1 << n_pts
    n_achievable = sum(1 for _, a in results if a)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # ── Left: threshold function on the real line ─────────────────────────────
    ax = axes[0]
    xs = np.linspace(-0.5, n_pts - 0.5 + 1, 500)
    fx = np.where(xs >= threshold, 1, -1)
    ax.plot(xs, fx, color='steelblue', lw=2, label=f'$f_t(x)=\\text{{sign}}(x-t)$,  $t={threshold:.1f}$')
    ax.axvline(threshold, color='darkorange', ls='--', lw=1.8, label=f'Threshold $t={threshold:.1f}$')
    for i in range(n_pts):
        label_i = +1 if i >= threshold else -1
        color_i = 'limegreen' if label_i == +1 else 'firebrick'
        ax.plot(i, label_i, 'o', ms=14, color=color_i, zorder=5,
                markeredgecolor='k', markeredgewidth=0.8)
        ax.text(i, label_i + 0.18, f'$x_{i+1}$', ha='center', fontsize=11)
    ax.set_xlim(-0.5, n_pts - 0.5 + 1)
    ax.set_ylim(-1.6, 1.6)
    ax.set_yticks([-1, 1])
    ax.set_yticklabels([r'$-1$', r'$+1$'])
    ax.set_xlabel('$x$', fontsize=13)
    ax.set_title(f'Threshold function on $n={n_pts}$ points\n'
                 f'VC dim = 1: can shatter 1 point, not 2', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2, axis='y')

    # ── Right: achievability grid ─────────────────────────────────────────────
    ax2 = axes[1]
    mat  = np.array([[(+1 if (mask >> i) & 1 else -1) for i in range(n_pts)]
                     for mask in range(n_total)])
    achv = np.array([1.0 if a else 0.0 for _, a in results])

    cmap = mcolors.ListedColormap(['firebrick', 'limegreen'])
    im   = ax2.imshow(achv[:, np.newaxis], cmap=cmap, vmin=0, vmax=1,
                      aspect='auto', extent=[-0.5, 0.5, -0.5, n_total - 0.5])
    for mask, (labels, achv_i) in enumerate(results):
        lab_str = '  '.join('+' if l == 1 else '-' for l in labels)
        ax2.text(0, mask, lab_str, ha='center', va='center',
                 fontsize=11, color='white', fontweight='bold')

    ax2.set_yticks([])
    ax2.set_xticks([])
    ax2.set_title(
        f'All $2^{n_pts} = {n_total}$ labelings\n'
        f'Green = achievable by threshold  ({n_achievable}/{n_total})',
        fontsize=11,
    )
    p_green = mpatches.Patch(color='limegreen', label=f'Achievable ({n_achievable})')
    p_red   = mpatches.Patch(color='firebrick',  label=f'Not achievable ({n_total-n_achievable})')
    ax2.legend(handles=[p_green, p_red], fontsize=9, loc='lower right',
               bbox_to_anchor=(1.55, 0))

    plt.tight_layout()
    plt.show()


## Linear classifiers in $\mathbb{R}^2$: VC dimension 3

A linear classifier in $\mathbb{R}^2$ is determined by a half-plane:
$$f_{w,b}(x) = \operatorname{sign}(w_1 x_1 + w_2 x_2 + b).$$

**Claim**: any 3 non-collinear points can be shattered by half-planes.
*Proof*: for any labeling, the linear programming test below finds a separating line.

**Claim**: no 4 points (in any configuration) can be shattered by half-planes.
*Proof*: by Radon's theorem, any 4 points in $\mathbb{R}^2$ can be partitioned into
two sets whose convex hulls intersect; the corresponding labeling $(+,-,+,-)$
has no linear separator.

**Therefore**: $\operatorname{VC-dim}(\text{half-planes in } \mathbb{R}^2) = 3$.

Use the controls below to cycle through all $2^n$ labelings and see which are linearly separable.


In [ ]:
POINT_CONFIGS = {
    '3 non-collinear': np.array([[0.0, 0.0], [1.0, 0.0], [0.5, 1.0]]),
    '4 in convex position (square)': np.array([[0.0,0.0],[1.0,0.0],[1.0,1.0],[0.0,1.0]]),
    '4 with one interior': np.array([[0.0,0.0],[1.0,0.0],[0.5,1.0],[0.5,0.4]]),
}

@widgets.interact(
    config=widgets.Dropdown(
        options=list(POINT_CONFIGS.keys()),
        description='Point configuration',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px'),
    ),
    labeling_idx=widgets.IntSlider(
        value=0, min=0, max=15, step=1,
        description='Labeling index',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px'),
    ),
    show_all=widgets.Checkbox(value=True,
        description='Show all labelings (right panel)'),
)
def linear_2d_plot(config, labeling_idx, show_all):
    pts = POINT_CONFIGS[config]
    n   = len(pts)
    mask = labeling_idx % (1 << n)
    labels = np.array([+1 if (mask >> i) & 1 else -1 for i in range(n)])

    sep, w, b = is_separable(pts, labels)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    # ── Left: current labeling ────────────────────────────────────────────────
    ax = axes[0]
    pad = 0.4
    xlim = [pts[:,0].min() - pad, pts[:,0].max() + pad]
    ylim = [pts[:,1].min() - pad, pts[:,1].max() + pad]

    # Background shading
    if sep:
        xx, yy = np.meshgrid(np.linspace(*xlim, 200), np.linspace(*ylim, 200))
        bg = np.sign(w[0]*xx + w[1]*yy + b)
        ax.contourf(xx, yy, bg, levels=[-2, 0, 2], colors=['#ffcccc','#ccffcc'],
                    alpha=0.35)
        lx, ly = separating_line_pts(w, b, xlim)
        in_ylim = (ly >= ylim[0]) & (ly <= ylim[1])
        if in_ylim.any():
            ax.plot(lx[in_ylim], ly[in_ylim], 'k-', lw=2,
                    label='Separating line')

    for i, (p, lbl) in enumerate(zip(pts, labels)):
        c = 'steelblue' if lbl == +1 else 'firebrick'
        m = 'o' if lbl == +1 else 's'
        ax.plot(p[0], p[1], m, ms=16, color=c, zorder=5,
                markeredgecolor='k', markeredgewidth=0.8)
        ax.text(p[0]+0.06, p[1]+0.06, f'$x_{i+1}$', fontsize=11, zorder=6)

    label_str = '(' + ', '.join('+' if l==1 else '-' for l in labels) + ')'
    status    = 'SEPARABLE' if sep else 'NOT separable'
    color_s   = 'limegreen' if sep else 'firebrick'
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_aspect('equal')
    ax.set_title(f'Labeling {labeling_idx}: {label_str}\n'
                 f'Status: {status}', fontsize=12, color=color_s)
    ax.set_xlabel('$x_1$', fontsize=12); ax.set_ylabel('$x_2$', fontsize=12)
    ax.grid(True, alpha=0.25)
    p_plus  = mpatches.Patch(color='steelblue', label='+1 (circle)')
    p_minus = mpatches.Patch(color='firebrick',  label='-1 (square)')
    ax.legend(handles=[p_plus, p_minus], fontsize=10)

    # ── Right: achievability matrix ───────────────────────────────────────────
    ax2 = axes[1]
    if show_all:
        n_lbls = 1 << n
        achv_vec = []
        for m2 in range(n_lbls):
            labs2 = np.array([+1 if (m2>>i)&1 else -1 for i in range(n)])
            s2, _, _ = is_separable(pts, labs2)
            achv_vec.append(1.0 if s2 else 0.0)
        achv_arr = np.array(achv_vec)
        n_sep    = int(achv_arr.sum())

        cmap2 = mcolors.ListedColormap(['firebrick', 'limegreen'])
        ax2.imshow(achv_arr[:, np.newaxis], cmap=cmap2, vmin=0, vmax=1,
                   aspect='auto', extent=[-0.5, 0.5, -0.5, n_lbls-0.5])
        for m2 in range(n_lbls):
            labs2 = [+1 if (m2>>i)&1 else -1 for i in range(n)]
            ls    = '(' + ','.join('+' if l==1 else '-' for l in labs2) + ')'
            ax2.text(0, m2, ls, ha='center', va='center',
                     fontsize=9 if n==4 else 11, color='white', fontweight='bold')
        # Highlight current row
        ax2.add_patch(plt.Rectangle((-0.5, mask-0.5), 1, 1,
                                     fill=False, edgecolor='gold', lw=3))
        ax2.set_yticks([]); ax2.set_xticks([])
        ax2.set_title(
            f'All $2^{n}={n_lbls}$ labelings  ({n_sep} separable)\n'
            'Gold border = currently shown; VC-dim = ' +
            ('3' if n==3 else '3 (not 4)'),
            fontsize=11,
        )
        p_g = mpatches.Patch(color='limegreen', label=f'Separable ({n_sep})')
        p_r = mpatches.Patch(color='firebrick',  label=f'Not separable ({n_lbls-n_sep})')
        ax2.legend(handles=[p_g,p_r], fontsize=9, loc='lower right',
                   bbox_to_anchor=(1.7, 0))
    else:
        ax2.text(0.5, 0.5, 'Uncheck "Show all"\nto see the achievability matrix',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=12)
        ax2.set_axis_off()

    plt.tight_layout()
    plt.show()


## The Fundamental Theorem of PAC Learning

**PAC (Probably Approximately Correct) learnability**: $\mathcal{F}$ is PAC-learnable if
there exists an algorithm that, given $m$ i.i.d. samples from any distribution $\mu$,
outputs a hypothesis with error $\leq \varepsilon$ with probability $\geq 1 - \delta$,
using only $m = m(\varepsilon, \delta)$ samples independent of $\mu$.

**Theorem** (Blumer, Ehrenfeucht, Haussler, Warmuth 1989):
$$\mathcal{F} \text{ is PAC-learnable} \;\iff\; \operatorname{VC-dim}(\mathcal{F}) < \infty.$$
Moreover, the **sample complexity** satisfies
$$m(\varepsilon, \delta) = \Theta\!\left(\frac{\operatorname{VC-dim}(\mathcal{F}) + \log(1/\delta)}{\varepsilon}\right).$$

**Connection to NIP**: finite VC dimension is exactly the NIP condition
for $\{0,1\}$-valued function classes.
For the $\mathbb{R}$-valued functions in the research papers, NIP generalizes this:
the independence pattern (IP) is the appropriate infinite-dimensional analogue of shattering.

**Corollary for deep computations**: a tame (NIP) deep equilibrium defines a
PAC-learnable function class; a wild (IP) equilibrium may not be learnable at any sample size.


In [ ]:
# Independence matrices: NIP (intervals) vs IP (all binary functions)
# Rows = test points; columns = functions; entry = f(x)

N_PTS   = 5
test_xs = np.linspace(0.1, 0.9, N_PTS)

# ── NIP: interval class [a,b] ─────────────────────────────────────────────────
# Generate all distinct interval patterns
a_list, b_list, labels_iv = [], [], []
for i in range(N_PTS):
    for j in range(i, N_PTS):
        a_list.append(test_xs[i] - 0.001)
        b_list.append(test_xs[j] + 0.001)
        labels_iv.append(f'[{test_xs[i]:.2f},{test_xs[j]:.2f}]')
# Add the empty interval (all -1)
a_list.append(10.0); b_list.append(10.0); labels_iv.append('(empty)')
M_nip = interval_matrix(test_xs, a_list, b_list)

# ── IP: all binary functions on N_PTS points ──────────────────────────────────
M_ip  = all_binary_matrix(N_PTS)    # shape (N_PTS, 2^N_PTS)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5),
                         gridspec_kw={'width_ratios': [len(a_list), M_ip.shape[1]]})

cmap_bin = mcolors.ListedColormap(['#d73027', '#4dac26'])   # red=0, green=1

for ax, M, title, col_labels in [
    (axes[0], M_nip,
     f'NIP class: intervals $[a,b]$ on $[0,1]$\n'
     f'VC dim = 2: each column is a contiguous block of 1s',
     labels_iv),
    (axes[1], M_ip,
     f'IP class: all $2^{N_PTS}={1<<N_PTS}$ binary functions on {N_PTS} points\n'
     f'VC dim = $\\infty$: every 0/1 column pattern appears',
     [f'{m:0{N_PTS}b}' for m in range(1 << N_PTS)]),
]:
    im = ax.imshow(M, cmap=cmap_bin, vmin=0, vmax=1, aspect='auto',
                   interpolation='nearest')
    ax.set_yticks(range(N_PTS))
    ax.set_yticklabels([f'$x_{i+1}={test_xs[i]:.2f}$' for i in range(N_PTS)],
                        fontsize=10)
    ax.set_xticks(range(M.shape[1]))
    ax.set_xticklabels(col_labels, rotation=90, fontsize=7 if M.shape[1] > 10 else 9)
    ax.set_xlabel('Function (column = labeling pattern)', fontsize=11)
    ax.set_title(title, fontsize=11)

# Annotate the "forbidden" pattern (+,-,+) in the NIP matrix
forbidden_pattern = np.array([1.0, 0.0, 1.0, 0.0, 0.0])
for j in range(M_nip.shape[1]):
    if np.allclose(M_nip[:, j], forbidden_pattern):
        axes[0].axvline(j, color='gold', lw=3, alpha=0.8, label='Forbidden pattern')

# Highlight column with pattern (+,-,+) for first 3 points in IP matrix
for j in range(M_ip.shape[1]):
    if M_ip[0,j]==1 and M_ip[1,j]==0 and M_ip[2,j]==1:
        axes[1].axvline(j, color='gold', lw=2, alpha=0.6,
                        label='$(+,-,+,...)$ pattern (exists in IP)')
        break

for ax in axes:
    ax.legend(fontsize=9, loc='lower right')

plt.tight_layout()
plt.show()

print('NIP (intervals): the pattern (+,-,+) for points x1 < x2 < x3')
print('  is IMPOSSIBLE — no interval contains x1 and x3 but not x2.')
print()
print('IP (all binary): every 0/1 column pattern appears — any set')
print('  of n points can be shattered. This is the Independence Property.')


In [ ]:
# Sample complexity of PAC learning as a function of VC dimension

eps_vals   = [0.10, 0.05, 0.01]
delta      = 0.05
vc_dims    = np.arange(1, 51)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: sample complexity vs VC dimension ───────────────────────────────────
ax = axes[0]
for eps in eps_vals:
    # m(eps, delta) ~ (VC-dim + log(1/delta)) / eps   (Blumer et al. upper bound)
    m_upper = (vc_dims + np.log(1.0 / delta)) / eps
    ax.semilogy(vc_dims, m_upper, lw=2, label=f'$\\varepsilon={eps}$')

# Annotate known function classes
annotations = [
    (1,  'Threshold\n(d=1)'),
    (2,  'Intervals\n(d=2)'),
    (3,  'Half-planes\n(d=3)'),
    (10, 'Small NN\n(d~10)'),
    (30, 'Larger NN\n(d~30)'),
]
for d, name in annotations:
    m_ann = (d + np.log(1/delta)) / 0.05
    ax.annotate(name, xy=(d, m_ann), xytext=(d+2, m_ann*2),
                fontsize=8, color='gray',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_xlabel('VC dimension $d$', fontsize=13)
ax.set_ylabel('Sample complexity $m(\\varepsilon, \\delta=0.05)$', fontsize=12)
ax.set_title('PAC sample complexity grows linearly in VC dim\n'
             '(log scale on y-axis)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)

# ── Right: "learnability frontier" in (VC dim, sample size) space ─────────────
ax2 = axes[1]
eps  = 0.05
m_grid = np.logspace(1, 5, 300)
d_grid = np.arange(1, 100)
MM, DD = np.meshgrid(m_grid, d_grid)
# Region: m >= (d + log(1/delta)) / eps  => PAC-learnable with m samples
learnable = (MM >= (DD + np.log(1/delta)) / eps).astype(float)
ax2.contourf(m_grid, d_grid, learnable, levels=[0.5, 1.5],
             colors=['#ffe0e0', '#d4edda'])
ax2.contour(m_grid, d_grid, learnable, levels=[0.5], colors='black', linewidths=1.5)
ax2.set_xscale('log')
ax2.set_xlabel('Training sample size $m$', fontsize=13)
ax2.set_ylabel('VC dimension $d$', fontsize=13)
ax2.set_title(
    'PAC learnability frontier ($\\varepsilon=0.05$, $\\delta=0.05$)\n'
    'Green = learnable with m samples; Red = insufficient data',
    fontsize=11,
)
ax2.text(200, 20, 'Learnable\n(NIP / finite VC dim)', fontsize=11,
         color='#155724', ha='center')
ax2.text(30, 70, 'Insufficient\ndata', fontsize=11,
         color='#721c24', ha='center')

plt.tight_layout()
plt.show()

print(f'Summary: VC dim d=1 (thresholds) needs ~{int((1+np.log(20))/0.05)} samples')
print(f'         VC dim d=3 (half-planes) needs ~{int((3+np.log(20))/0.05)} samples')
print(f'         VC dim d=infinity => NO finite sample size suffices (IP / wild)')


## Contemporary AI: in-context learning and VC dimension

Large language models exhibit **in-context learning** (ICL):
given a prompt containing input-output examples, the model predicts new outputs
*without updating its weights*.

**Theoretical view** (Garg et al. 2022; Akyürek et al. 2023):
transformers can implement learning algorithms in their forward pass —
essentially, a transformer of depth $L$ and width $d$ can
*simulate* any learning algorithm with sample complexity $O(L \cdot d)$.

**The VC-dimension connection**: if the target function class has VC dimension $k$,
a transformer needs $O(k)$ in-context examples to learn it.
For function classes with infinite VC dimension (IP), no fixed-depth transformer
can learn them reliably from bounded context.

**Open question**: which function classes are "ICL-learnable" by transformers of fixed size?
The answer involves a combination of VC theory (as above) and the
deep-computation framework: the ICL target must define a *tame* (NIP) class.

The NIP/VC framework thus appears at the intersection of:
- Classical PAC learning theory (Valiant 1984)
- Model-theoretic stability (Shelah 1970s)
- Modern transformer in-context learning (2022–2025)
